# 10 – Hyperparameter Tuning: ALS Cross-Validation

Tunes the ALS recommender from step 09 using **k-fold cross-validation** over a configurable parameter grid:
- `rank`: number of latent factors
- `reg_param`: L2 regularisation strength
- `max_iter`: optimisation iterations

The best model (lowest CV RMSE) is selected, evaluated on the held-out test set, and its recommendations overwrite `gold.recommendations`.

**Source:** `gold.book_ratings`
**Target:** `gold.recommendations`
**Config:** `config/10_hyperparameter_tuning.json`

In [0]:
import json, sys, importlib

SETUP_CONFIG = "/Workspace/Users/tharcisio.leone@x1f-fink.one/poc/config/00_setup.json"
with open(SETUP_CONFIG) as f:
    setup = json.load(f)

sys.path.insert(0, setup["project_path"])

for mod in [m for m in list(sys.modules.keys()) if m.startswith("source")]:
    del sys.modules[mod]

HyperparameterTuning = importlib.import_module("source.10_hyperparameter_tuning").HyperparameterTuning

CONFIG_PATH = f"{setup['project_path']}/config/10_hyperparameter_tuning.json"

In [0]:
pipeline = HyperparameterTuning(config_path=CONFIG_PATH)
results = pipeline.run()

print(f"\n=== Best Parameters ===")
for param, value in results["best_params"].items():
    print(f"  {param}: {value}")

print(f"\n=== Test Set Metrics ===")
for metric, value in results["metrics"].items():
    print(f"  {metric}: {value}")

In [0]:
import pandas as pd

cv_df = pd.DataFrame(results["cv_results"]).sort_values("avg_rmse")
print(f"Grid search: {len(cv_df)} parameter combinations")
print(f"Best CV RMSE: {cv_df['avg_rmse'].min()}")
print()
display(cv_df.head(20))

In [0]:
display(results["recommendations"])

In [0]:
from pyspark.sql import functions as F

sample_user = results["recommendations"].select("user_id").first()[0]
user_recs = results["recommendations"].filter(F.col("user_id") == sample_user)
print(f"\nTop recommendations for user {sample_user}:")
display(user_recs)